In [8]:
#Cuda Program : Matrix Multifplication using CUDA C
%%writefile cuda2.cu
#include <iostream>
#include <cuda_runtime.h>

using namespace std;

// CUDA Kernel for Matrix Multiplication
__global__ void matrixMultiply(int *A, int *B, int *C,
                               int rowsA, int colsA, int colsB) {

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < rowsA && col < colsB) {

        int sum = 0;

        for(int k = 0; k < colsA; k++) {
            sum += A[row * colsA + k] * B[k * colsB + col];
        }

        C[row * colsB + col] = sum;
    }
}

int main() {

    int rowsA, colsA, rowsB, colsB;

    cout << "Enter rows and columns of Matrix A: ";
    cin >> rowsA >> colsA;

    cout << "Enter rows and columns of Matrix B: ";
    cin >> rowsB >> colsB;

    // Matrix multiplication condition
    if(colsA != rowsB) {
        cout << "Matrix multiplication not possible!\n";
        return 0;
    }

    // Host matrices
    int *h_A = new int[rowsA * colsA];
    int *h_B = new int[rowsB * colsB];
    int *h_C = new int[rowsA * colsB];

    cout << "\nEnter elements of Matrix A:\n";

    for(int i = 0; i < rowsA; i++) {
        for(int j = 0; j < colsA; j++) {
            cin >> h_A[i * colsA + j];
        }
    }

    cout << "\nEnter elements of Matrix B:\n";

    for(int i = 0; i < rowsB; i++) {
        for(int j = 0; j < colsB; j++) {
            cin >> h_B[i * colsB + j];
        }
    }

    // Device pointers
    int *d_A, *d_B, *d_C;

    // Memory sizes
    int sizeA = rowsA * colsA * sizeof(int);
    int sizeB = rowsB * colsB * sizeof(int);
    int sizeC = rowsA * colsB * sizeof(int);

    // Allocate GPU memory
    cudaMalloc((void**)&d_A, sizeA);
    cudaMalloc((void**)&d_B, sizeB);
    cudaMalloc((void**)&d_C, sizeC);

    // Copy data CPU -> GPU
    cudaMemcpy(d_A, h_A, sizeA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, sizeB, cudaMemcpyHostToDevice);

    // Define block size
    dim3 threadsPerBlock(16, 16);

    // Define grid size
    dim3 blocksPerGrid(
        (colsB + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (rowsA + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    // Launch kernel
    matrixMultiply<<<blocksPerGrid, threadsPerBlock>>>(
        d_A, d_B, d_C,
        rowsA, colsA, colsB
    );

    // Copy result GPU -> CPU
    cudaMemcpy(h_C, d_C, sizeC, cudaMemcpyDeviceToHost);

    // Display result
    cout << "\nResultant Matrix:\n";

    for(int i = 0; i < rowsA; i++) {

        for(int j = 0; j < colsB; j++) {
            cout << h_C[i * colsB + j] << " ";
        }

        cout << endl;
    }

    // Free GPU memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    // Free CPU memory
    delete[] h_A;
    delete[] h_B;
    delete[] h_C;

    return 0;
}

Writing cuda2.cu


In [11]:
!nvcc cuda2.cu -o output

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [13]:
!./output

Enter rows and columns of Matrix A: 2 2
Enter rows and columns of Matrix B: 2 2

Enter elements of Matrix A:
1 2
3 4

Enter elements of Matrix B:
5 6
7 8

Resultant Matrix:
19 22 
43 50 
